In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "database" / "sonoran_cycles.db"

print(DB_PATH)
print(DB_PATH.exists())

/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/database/sonoran_cycles.db
True


In [2]:
def run_query(query):
    """
    Runs a SQL query against the Sonoran Cycles SQLite database.
    """

    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(query, conn)

In [3]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY
    name;
"""

run_query(query)

,table_name
0,calendar
1,customers
2,daily_kpi_summary
3,daily_order_summary
4,forecast_accuracy_by_model
5,forecast_history
6,inventory_history
7,inventory_kpi_summary
8,model_performance_summary
9,monthly_sales_summary


In [4]:
query = """
SELECT
    model_name,
    category,
    SUM(requested_qty) AS requested_units,
    SUM(fulfilled_qty) AS fulfilled_units,
    SUM(backordered_qty) AS backordered_units,
    ROUND(SUM(extended_price), 2) AS booked_revenue,
    ROUND(SUM(fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(
        CAST(SUM(fulfilled_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS service_level,
    ROUND(
        CAST(SUM(backordered_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS backorder_rate
FROM sales_order_lines
GROUP BY
    model_name,
    category
ORDER BY
    booked_revenue DESC;
"""

model_performance = run_query(query)
model_performance

,model_name,category,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,service_level,backorder_rate
0,Romero,Aggressive Trail,51613,42035,9578,1.715949e+08,1.396967e+08,0.814,0.186
1,Sabino,Trail,54391,43932,10459,1.640263e+08,1.323868e+08,0.808,0.192
2,Oracle,Enduro,35759,32535,3224,1.411246e+08,1.283495e+08,0.910,0.090
3,Catalina,Cross Country,28566,24962,3604,6.280795e+07,5.486534e+07,0.874,0.126
4,Sky Island,eMTB,11986,11938,48,5.876006e+07,5.852482e+07,0.996,0.004
5,Rincon,Downcountry,16454,15685,769,4.181898e+07,3.987979e+07,0.953,0.047
6,Sonoita,Gravel,7605,7587,18,1.855156e+07,1.850667e+07,0.998,0.002


In [5]:
query = """
SELECT
    so.sales_channel,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    COUNT(DISTINCT so.customer_id) AS customers,
    SUM(sol.requested_qty) AS requested_units,
    SUM(sol.fulfilled_qty) AS fulfilled_units,
    SUM(sol.backordered_qty) AS backordered_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(AVG(sol.unit_price), 2) AS average_unit_price,
    ROUND(
        CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS service_level
FROM sales_orders so
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
GROUP BY
    so.sales_channel
ORDER BY
    booked_revenue DESC;
"""

channel_performance = run_query(query)
channel_performance

,sales_channel,sales_orders,customers,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,average_unit_price,service_level
0,Dealer,21458,35,188734,163425,25309,5.750124e+08,4.996065e+08,3045.47,0.866
1,DTC,15542,1,17640,15249,2391,8.367206e+07,7.260315e+07,4743.43,0.864


In [6]:
query = """
SELECT
    substr(so.order_date, 1, 7) AS order_month,
    so.sales_channel,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    SUM(sol.requested_qty) AS requested_units,
    SUM(sol.fulfilled_qty) AS fulfilled_units,
    SUM(sol.backordered_qty) AS backordered_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(
        CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS service_level
FROM sales_orders so
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
GROUP BY
    substr(so.order_date, 1, 7),
    so.sales_channel
ORDER BY
    order_month,
    so.sales_channel;
"""

monthly_trend = run_query(query)
monthly_trend.head()

,order_month,sales_channel,sales_orders,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,service_level
0,2022-01,DTC,195,219,216,3,1002481.00,987484.00,0.986
1,2022-01,Dealer,264,2334,2309,25,7080648.66,6999812.28,0.989
2,2022-02,DTC,210,234,214,20,1090566.00,1003286.00,0.915
3,2022-02,Dealer,289,2615,2414,201,7951322.68,7354785.71,0.923
4,2022-03,DTC,311,363,314,49,1776137.00,1536186.00,0.865


In [7]:
query = """
SELECT
    sol.product_id,
    sol.sku,
    sol.model_name,
    sol.category,
    sol.size,
    sol.color,
    SUM(sol.requested_qty) AS requested_units,
    SUM(sol.fulfilled_qty) AS fulfilled_units,
    SUM(sol.backordered_qty) AS backordered_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(
        CAST(SUM(sol.backordered_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS backorder_rate
FROM sales_order_lines sol
GROUP BY
    sol.product_id,
    sol.sku,
    sol.model_name,
    sol.category,
    sol.size,
    sol.color
HAVING
    SUM(sol.requested_qty) > 0
ORDER BY
    backordered_units DESC,
    backorder_rate DESC
LIMIT 25;
"""

worst_backorder_skus = run_query(query)
worst_backorder_skus

,product_id,sku,model_name,category,size,color,requested_units,fulfilled_units,backordered_units,booked_revenue,backorder_rate
0,1103,SON-ROM-C-M-IR,Romero,Aggressive Trail,M,Ironwood,3100,2108,992,10304989.89,0.320
1,1073,SON-SAB-C-M-IR,Sabino,Trail,M,Ironwood,3188,2209,979,9616003.11,0.307
2,1074,SON-SAB-C-M-SG,Sabino,Trail,M,Saguaro,3105,2180,925,9312285.15,0.298
3,1076,SON-SAB-C-M-GR,Sabino,Trail,M,Granite,3153,2228,925,9525448.80,0.293
4,1077,SON-SAB-C-M-CY,Sabino,Trail,M,Coyote,3188,2282,906,9653025.06,0.284
5,1105,SON-ROM-C-M-SS,Romero,Aggressive Trail,M,Sunset,3041,2138,903,10088191.59,0.297
6,1075,SON-SAB-C-M-SS,Sabino,Trail,M,Sunset,3049,2162,887,9195516.54,0.291
7,1104,SON-ROM-C-M-SG,Romero,Aggressive Trail,M,Saguaro,3026,2154,872,10121985.09,0.288
8,1106,SON-ROM-C-M-GR,Romero,Aggressive Trail,M,Granite,2998,2134,864,9965287.23,0.288
9,1107,SON-ROM-C-M-CY,Romero,Aggressive Trail,M,Coyote,3018,2187,831,10052994.36,0.275


In [8]:
query = """
SELECT
    product_id,
    sku,
    model_name,
    category,
    size,
    color,
    ROUND(average_available, 2) AS average_available,
    minimum_available,
    ending_available,
    stockout_days,
    ROUND(stockout_rate, 3) AS stockout_rate
FROM inventory_kpi_summary
ORDER BY
    stockout_days DESC,
    stockout_rate DESC,
    average_available ASC
LIMIT 25;
"""

worst_stockout_skus = run_query(query)
worst_stockout_skus

,product_id,sku,model_name,category,size,color,average_available,minimum_available,ending_available,stockout_days,stockout_rate
0,1106,SON-ROM-C-M-GR,Romero,Aggressive Trail,M,Granite,22.79,0,56,416,0.285
1,1074,SON-SAB-C-M-SG,Sabino,Trail,M,Saguaro,22.95,0,47,416,0.285
2,1073,SON-SAB-C-M-IR,Sabino,Trail,M,Ironwood,22.02,0,5,413,0.283
3,1103,SON-ROM-C-M-IR,Romero,Aggressive Trail,M,Ironwood,23.10,0,60,405,0.277
4,1076,SON-SAB-C-M-GR,Sabino,Trail,M,Granite,22.66,0,35,403,0.276
5,1077,SON-SAB-C-M-CY,Sabino,Trail,M,Coyote,21.83,0,53,399,0.273
6,1075,SON-SAB-C-M-SS,Sabino,Trail,M,Sunset,23.00,0,49,397,0.272
7,1107,SON-ROM-C-M-CY,Romero,Aggressive Trail,M,Coyote,23.48,0,22,388,0.266
8,1105,SON-ROM-C-M-SS,Romero,Aggressive Trail,M,Sunset,23.12,0,58,385,0.264
9,1104,SON-ROM-C-M-SG,Romero,Aggressive Trail,M,Saguaro,23.45,0,0,373,0.255


In [9]:
query = """
SELECT
    supplier_id,
    supplier_name,
    purchase_orders,
    open_purchase_orders,
    received_purchase_orders,
    ordered_units,
    received_units,
    open_units,
    ROUND(average_lead_time_days, 1) AS average_lead_time_days,
    ROUND(receipt_rate, 3) AS receipt_rate
FROM supplier_performance_summary
ORDER BY
    open_units DESC,
    ordered_units DESC;
"""

supplier_exposure = run_query(query)
supplier_exposure

,supplier_id,supplier_name,purchase_orders,open_purchase_orders,received_purchase_orders,ordered_units,received_units,open_units,average_lead_time_days,receipt_rate
0,S001,Fox Factory,2141,15,2126,131963,131038,925,21.0,0.993
1,S002,SRAM,805,10,795,49261,48647,614,30.0,0.988


In [10]:
query = """
SELECT
    model_name,
    category,
    actual_qty,
    forecast_qty,
    absolute_error,
    forecast_error,
    ROUND(wape, 3) AS wape,
    ROUND(bias_pct, 3) AS bias_pct,
    ROUND(forecast_accuracy, 3) AS forecast_accuracy
FROM forecast_accuracy_by_model
ORDER BY
    wape DESC;
"""

forecast_accuracy = run_query(query)
forecast_accuracy

,model_name,category,actual_qty,forecast_qty,absolute_error,forecast_error,wape,bias_pct,forecast_accuracy
0,Sonoita,Gravel,7223,7168,4265,55,0.590,0.008,0.410
1,Sky Island,eMTB,11465,11423,5604,42,0.489,0.004,0.511
2,Rincon,Downcountry,15837,15550,6531,287,0.412,0.018,0.588
3,Catalina,Cross Country,27315,27072,8591,243,0.315,0.009,0.685
4,Oracle,Enduro,34125,34056,9411,69,0.276,0.002,0.724
5,Romero,Aggressive Trail,49115,49287,11798,-172,0.240,-0.004,0.760
6,Sabino,Trail,51722,51790,12128,-68,0.234,-0.001,0.766
